# Metro-ASR — Streaming & Server Deployment

Three ways to serve Metro-ASR beyond a single `engine.transcribe()` call:
1. **REST API** — the Flask server from `scripts/serve.py`, called from Python, curl, or Postman
2. **Streaming** — chunked transcription over a Python generator (e.g. a growing audio buffer)
3. **Gradio streaming UI** — the microphone tab from `app.py`, with a public link

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MohammedAly22/metro-asr/blob/main/examples/streaming_server.ipynb)


## Setup

Clones the repo (needed for `scripts/serve.py` and `app.py`) and downloads a demo clip.

In [ ]:
!pip install -q "metro-asr[server]"
!git clone -q https://github.com/MohammedAly22/metro-asr.git
%cd metro-asr
!cp test_samples/5.wav audio.wav
print("Ready: audio.wav")


## Option 1 — REST API server

The server needs to run *while* the cells below query it. In a normal terminal that means
two windows; in a notebook it means launching it as a **background process** from Python and
polling `/health` until it responds, which is what the next cell does.

`METRO_MODEL=small` matters here for the same reason as in the other notebooks: there is no
local `checkpoints/` directory in a fresh Colab session, so the server is told to fetch the
model from HuggingFace instead. Because the server and the client cells below both run inside
the *same* Colab VM, plain `http://localhost:8000` is directly reachable — no tunnel needed for
this part (that's only for exposing a UI to a browser outside the VM, as in the Gradio notebook).

In [ ]:
import os, subprocess, time, requests

env = os.environ.copy()
env["METRO_MODEL"] = "small"
env["METRO_LM"] = "auto"       # drop this line for a faster, greedy-only start
env["METRO_PORT"] = "8000"

log = open("serve.log", "w")
server = subprocess.Popen(
    ["python", "scripts/serve.py"],
    env=env, stdout=log, stderr=subprocess.STDOUT,
)

SERVER_URL = "http://localhost:8000"
for _ in range(120):  # up to ~2 minutes for the model (and optionally the LM) to download + load
    try:
        if requests.get(f"{SERVER_URL}/health", timeout=2).ok:
            print("Server is up.")
            break
    except requests.exceptions.ConnectionError:
        pass
    time.sleep(1)
else:
    print("Server did not come up in time — check serve.log:")
    print(open("serve.log").read()[-2000:])


### Python client

In [ ]:
resp = requests.get(f"{SERVER_URL}/health")
print("Health:", resp.json())

resp = requests.get(f"{SERVER_URL}/info")
print("Info:  ", resp.json())


In [ ]:
with open("audio.wav", "rb") as f:
    resp = requests.post(f"{SERVER_URL}/transcribe", files={"audio": f})

result = resp.json()
print(f"Text:     {result['text']}")
print(f"Duration: {result['duration']}s")
print(f"RTF:      {result['rtf']}")


In [ ]:
# Beam search + language model (only if METRO_LM=auto was set above)
with open("audio.wav", "rb") as f:
    resp = requests.post(
        f"{SERVER_URL}/transcribe",
        files={"audio": f},
        data={"beam_search": "true"},
    )
print("Text:", resp.json()["text"])


In [ ]:
# Batch — reuses the same file twice for demonstration; pass distinct paths in practice
files = [("audio", open("audio.wav", "rb")), ("audio", open("audio.wav", "rb"))]
resp = requests.post(f"{SERVER_URL}/transcribe/batch", files=files)

for r in resp.json()["results"]:
    print(f"  {r['text']}  ({r['duration']}s)")


### curl

Only reachable from *inside* this Colab VM (e.g. a `!curl` cell here) — `localhost` doesn't
mean your own machine when the server is running on Google's infrastructure.

```bash
curl -X POST http://localhost:8000/transcribe -F "audio=@audio.wav"
curl -X POST http://localhost:8000/transcribe -F "audio=@audio.wav" -F "beam_search=true"
curl -X POST http://localhost:8000/transcribe/batch -F "audio=@a.wav" -F "audio=@b.wav"
curl http://localhost:8000/health
curl http://localhost:8000/info
```

### Postman

Postman runs on your own machine, so it can only reach this server if you swap `localhost`
for a public tunnel. Easiest path: run `scripts/serve.py` locally instead of in Colab, or add
your own tunnel (ngrok, cloudflared) in front of it.

In [ ]:
# Stop the background server when you're done with this section.
server.terminate()


## Option 2 — streaming transcription (Python API)

No server involved — `engine.transcribe_stream()` takes any generator that yields audio
chunks and returns transcriptions incrementally, e.g. for a microphone feed or a growing file.

`transcribe_stream` expects chunks already at the engine's sample rate (16 kHz); the generator
below resamples once up front so this works with a file recorded at any rate, not just clips
that already happen to be 16 kHz.

In [ ]:
import soundfile as sf
import torchaudio
import torch
from metro_asr import MetroASREngine

engine = MetroASREngine.from_pretrained("small")

def audio_chunk_generator(audio_path, chunk_seconds=5.0):
    """Yield fixed-size chunks at the engine's sample rate, simulating a live feed."""
    data, sr = sf.read(audio_path, dtype="float32")
    if data.ndim > 1:
        data = data.mean(axis=1)
    if sr != engine.sample_rate:
        data = torchaudio.functional.resample(
            torch.from_numpy(data), sr, engine.sample_rate
        ).numpy()
    chunk_size = int(engine.sample_rate * chunk_seconds)
    for i in range(0, len(data), chunk_size):
        yield data[i:i + chunk_size]

for chunk in engine.transcribe_stream(audio_chunk_generator("audio.wav")):
    print(f"[{chunk.total_duration:.1f}s] {chunk.text}")
    if chunk.is_final:
        print("--- end ---")


## Option 3 — Gradio streaming UI

The microphone-streaming tab lives in the same `app.py` as the file-upload demo (see
[gradio_app.ipynb](gradio_app.ipynb)) — there is no separate streaming script. `METRO_SHARE=true`
gets you a public link, since a Gradio mic widget needs the browser on your own machine to
reach the server, unlike the REST calls above which ran entirely inside this VM.

In [ ]:
!pip install -q "metro-asr[demo]"


In [ ]:
%env METRO_MODEL=small
%env METRO_SHARE=true
# Open the printed https://*.gradio.live link, then use the "Stream" tab.
!python app.py
